# 07 — Hooks, Session State & Compound Tools

This notebook demonstrates three primitives from the Core Uplift phase:

1. **HookRunner** — fire-and-forget lifecycle hooks that wrap tool execution with before/after/error callbacks. A broken hook never crashes the system.
2. **BaseSessionState** — serializable conversation state for saving and restoring sessions across process boundaries.
3. **CompoundTool** — the agent-decides, system-acts pattern: an agent produces structured output, an executor acts on it, and state is updated.

**Prerequisites:**
- A `.env` file in the project root (or environment variables) with:
  ```
  LLM_API_KEY=your_api_key
  LLM_MODEL=openai:gpt-4o   # or anthropic:claude-sonnet-4-20250514, google:gemini-2.0-flash, etc.
  ```

## 1. Setup

Load config and import the building blocks.

In [1]:
from orqest import load_config

config = load_config()
print(f"Model:    {config.llm_model}")
print(f"API key:  {config.llm_api_key[:8]}...")

Model:    openai:gpt-5.4
API key:  sk-proj-...


---

# Part 1: Lifecycle Hooks

Hooks let you observe tool execution without coupling observation logic to tool logic. They are **fire-and-forget**: if a hook raises, the error is logged at WARNING level and swallowed. The tool still completes normally.

## 2. Define hooks

In [2]:
import time
from typing import Any

from orqest.hooks import HookRunner


class LoggingHook:
    """Prints before/after/error events to stdout."""

    async def before_tool(self, tool_name: str, args: dict[str, Any], state: Any) -> None:
        print(f"  [LOG] >>> {tool_name} starting | args keys: {list(args.keys())}")

    async def after_tool(
        self, tool_name: str, args: dict[str, Any], result: Any, state: Any, duration_ms: float
    ) -> None:
        print(f"  [LOG] <<< {tool_name} completed in {duration_ms:.1f}ms")

    async def on_error(
        self, tool_name: str, args: dict[str, Any], error: Exception, state: Any
    ) -> None:
        print(f"  [LOG] !!! {tool_name} failed: {error}")


class MetricsHook:
    """Tracks execution count and total duration per tool."""

    def __init__(self) -> None:
        self.call_count: dict[str, int] = {}
        self.total_duration_ms: dict[str, float] = {}
        self.error_count: dict[str, int] = {}

    async def before_tool(self, tool_name: str, args: dict[str, Any], state: Any) -> None:
        self.call_count[tool_name] = self.call_count.get(tool_name, 0) + 1

    async def after_tool(
        self, tool_name: str, args: dict[str, Any], result: Any, state: Any, duration_ms: float
    ) -> None:
        self.total_duration_ms[tool_name] = self.total_duration_ms.get(tool_name, 0) + duration_ms

    async def on_error(
        self, tool_name: str, args: dict[str, Any], error: Exception, state: Any
    ) -> None:
        self.error_count[tool_name] = self.error_count.get(tool_name, 0) + 1

    def report(self) -> None:
        print("--- Metrics Report ---")
        for name in sorted(set(self.call_count) | set(self.error_count)):
            calls = self.call_count.get(name, 0)
            duration = self.total_duration_ms.get(name, 0)
            errors = self.error_count.get(name, 0)
            avg = duration / max(calls - errors, 1)
            print(f"  {name}: {calls} calls, {duration:.1f}ms total, {avg:.1f}ms avg, {errors} errors")


print("LoggingHook and MetricsHook defined.")

LoggingHook and MetricsHook defined.


## 3. Create HookRunner and demonstrate

The `HookRunner` dispatches events to all registered hooks. Let's fire some events manually to see the hooks in action.

In [3]:
logging_hook = LoggingHook()
metrics_hook = MetricsHook()

runner = HookRunner(hooks=[logging_hook, metrics_hook])

# Simulate a successful tool execution
print("Simulating a successful tool call:")
await runner.fire_before("fetch_data", {"url": "https://example.com"}, state=None)
await runner.fire_after("fetch_data", {"url": "https://example.com"}, result={"rows": 42}, state=None, duration_ms=150.3)

print()

# Simulate a failed tool execution
print("Simulating a failed tool call:")
await runner.fire_before("write_file", {"path": "/tmp/out.txt"}, state=None)
await runner.fire_error("write_file", {"path": "/tmp/out.txt"}, error=PermissionError("read-only filesystem"), state=None)

print()
metrics_hook.report()

Simulating a successful tool call:
  [LOG] >>> fetch_data starting | args keys: ['url']
  [LOG] <<< fetch_data completed in 150.3ms

Simulating a failed tool call:
  [LOG] >>> write_file starting | args keys: ['path']
  [LOG] !!! write_file failed: read-only filesystem

--- Metrics Report ---
  fetch_data: 1 calls, 150.3ms total, 150.3ms avg, 0 errors
  write_file: 1 calls, 0.0ms total, 0.0ms avg, 1 errors


## 4. Broken hook resilience

A critical design property: **a broken hook never crashes the system**. The `HookRunner._safe_call()` method catches any exception from a hook and logs it at WARNING level. The tool execution continues normally.

Let's prove it by adding a hook that always raises.

In [4]:
class BrokenHook:
    """A hook that always raises — simulates a buggy observer."""

    async def before_tool(self, tool_name: str, args: dict[str, Any], state: Any) -> None:
        raise RuntimeError("BrokenHook crashed on purpose!")

    async def after_tool(
        self, tool_name: str, args: dict[str, Any], result: Any, state: Any, duration_ms: float
    ) -> None:
        raise RuntimeError("BrokenHook crashed on purpose!")


# Create a runner with the broken hook sandwiched between good hooks
runner_with_broken = HookRunner(hooks=[logging_hook, BrokenHook(), metrics_hook])

print("Firing events with a broken hook in the middle:")
print("(The broken hook logs a warning but does NOT stop the other hooks)\n")

await runner_with_broken.fire_before("resilience_test", {"key": "value"}, state=None)
await runner_with_broken.fire_after("resilience_test", {"key": "value"}, result="ok", state=None, duration_ms=50.0)

print()
print("All hooks that come after BrokenHook still ran:")
metrics_hook.report()

2026-04-11 19:22:10.134 | WARNING  | orqest.hooks:_safe_call:120 - Hook BrokenHook.before_tool failed
2026-04-11 19:22:10.137 | WARNING  | orqest.hooks:_safe_call:120 - Hook BrokenHook.after_tool failed


Firing events with a broken hook in the middle:
(The broken hook logs a warning but does NOT stop the other hooks)

  [LOG] >>> resilience_test starting | args keys: ['key']
  [LOG] <<< resilience_test completed in 50.0ms

All hooks that come after BrokenHook still ran:
--- Metrics Report ---
  fetch_data: 1 calls, 150.3ms total, 150.3ms avg, 0 errors
  resilience_test: 1 calls, 50.0ms total, 50.0ms avg, 0 errors
  write_file: 1 calls, 0.0ms total, 0.0ms avg, 1 errors


### What's happening under the hood

1. `HookRunner` iterates over registered hooks and calls each method via `_safe_call()`
2. `_safe_call()` uses `getattr(hook, method_name, None)` — if the hook doesn't implement a method, it's silently skipped
3. The call is wrapped in `try/except Exception` — any error is logged with `loguru.logger.warning()` and swallowed
4. This means hooks are purely observational: they can never break the system they're observing

---

# Part 2: Session State Persistence

`BaseSessionState` extends `GlobalState` with a `session_id`, `created_at` timestamp, and JSON-safe serialization. The tricky part is that pydantic-ai `ModelMessage` objects are dataclasses (not Pydantic models), so they need special handling via `ModelMessagesTypeAdapter`.

## 5. Define a custom session state

In [5]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, BaseSessionState


class ProjectState(BaseSessionState):
    """Session state with project-specific fields."""

    project_name: str = "untitled"
    tags: list[str] = Field(default_factory=list)


class SummaryOutput(BaseModel):
    """Structured output from the summarization agent."""

    summary: str = Field(description="A concise summary of the user's message")
    key_points: list[str] = Field(description="Bullet points of the main ideas")


class SummaryAgent(BaseAgent[ProjectState, SummaryOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="summary_agent",
            system_prompt=(
                "You are a project research assistant. Summarize information "
                "concisely and extract key points. When following up on a previous "
                "topic, reference what was discussed before."
            ),
            output_type=SummaryOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: ProjectState, **kwargs) -> SummaryOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


print("ProjectState and SummaryAgent defined.")

ProjectState and SummaryAgent defined.


## 6. Run a multi-turn conversation

We'll run two turns against a real LLM, building up conversation history on our `ProjectState`.

In [6]:
agent = SummaryAgent(model=config.llm_model, api_key=config.llm_api_key)

state = ProjectState(project_name="solar-research", tags=["energy", "renewables"])

# Turn 1
state.add_message(
    "user",
    "Solar panel efficiency has improved from about 6% in the 1950s to over 47% "
    "in lab conditions today. Commercial panels typically achieve 20-23% efficiency. "
    "Perovskite solar cells are a promising new technology that could push commercial "
    "efficiency past 30% while being cheaper to manufacture than silicon cells."
)

output1 = await agent.run(state)

print(f"Project: {state.project_name}")
print(f"Session: {state.session_id[:8]}...")
print(f"Tags:    {state.tags}\n")
print("Turn 1 summary:")
print(output1.summary)
print(f"\nMessage history length: {len(state.message_history)}")

Project: solar-research
Session: 3b5cf06b...
Tags:    ['energy', 'renewables']

Turn 1 summary:
Solar panel efficiency has risen dramatically from roughly 6% in the 1950s to more than 47% in laboratory settings today, while commercial panels usually operate at 20–23%. Perovskite solar cells are highlighted as a promising emerging technology that may boost commercial efficiency beyond 30% at lower manufacturing cost than silicon.

Message history length: 3


In [7]:
# Turn 2 — follow-up referencing the first turn
state.add_message(
    "user",
    "Now focus specifically on perovskite solar cells. What are the key challenges "
    "preventing them from reaching commercial scale?"
)

output2 = await agent.run(state)

print("Turn 2 summary:")
print(output2.summary)
print("\nKey points:")
for point in output2.key_points:
    print(f"  - {point}")
print(f"\nMessage history length: {len(state.message_history)}")

Turn 2 summary:
Building on the earlier discussion of perovskites as a high-efficiency, lower-cost alternative to silicon, their main barriers to commercial scale are durability, scalable manufacturing, toxicity concerns, and long-term bankability.

Key points:
  - Stability is the biggest issue: perovskite cells degrade when exposed to moisture, oxygen, heat, UV light, and electrical stress.
  - Operational lifetime still lags silicon, making it hard to guarantee 20–30 years of field performance.
  - Manufacturing at scale is challenging: lab results are hard to reproduce uniformly over large areas with high yield.
  - Defect control and encapsulation remain critical because small material imperfections can sharply reduce performance and lifetime.
  - Many high-performing perovskites contain lead, creating environmental, regulatory, and recycling concerns.
  - Supply-chain maturity, standards, certification, and investor confidence are still underdeveloped compared with silicon.
  - I

## 7. Serialize state to JSON

`BaseSessionState.serialize()` produces a JSON-safe dict. The key challenge is that `message_history` contains pydantic-ai `ModelMessage` dataclass objects, which Pydantic cannot natively dump. The serialization uses `ModelMessagesTypeAdapter` to handle this correctly.

In [8]:
import json

serialized = state.serialize()

# Pretty-print the serialized state (truncate message_history for readability)
display_data = dict(serialized)
display_data["message_history"] = f"[{len(serialized['message_history'])} messages — truncated for display]"

print("Serialized state:")
print(json.dumps(display_data, indent=2, default=str))

# Show a snippet of the raw message_history to prove it's JSON-safe
print("\nFirst message_history entry (raw JSON):")
print(json.dumps(serialized["message_history"][0], indent=2, default=str)[:500])

Serialized state:
{
  "messages": [
    {
      "role": "user",
      "content": "Solar panel efficiency has improved from about 6% in the 1950s to over 47% in lab conditions today. Commercial panels typically achieve 20-23% efficiency. Perovskite solar cells are a promising new technology that could push commercial efficiency past 30% while being cheaper to manufacture than silicon cells."
    },
    {
      "role": "user",
      "content": "Now focus specifically on perovskite solar cells. What are the key challenges preventing them from reaching commercial scale?"
    }
  ],
  "session_id": "3b5cf06b-b87d-4834-ad7c-5e028fa57ff8",
  "created_at": "2026-04-11 19:22:12.235782",
  "project_name": "solar-research",
  "tags": [
    "energy",
    "renewables"
  ],
  "message_history": "[6 messages \u2014 truncated for display]"
}

First message_history entry (raw JSON):
{
  "parts": [
    {
      "content": "You are a project research assistant. Summarize information concisely and extract 

## 8. Deserialize and continue the conversation

Simulate a new process: deserialize from the JSON dict, then continue the conversation. The agent remembers everything from the previous session.

In [9]:
# Simulate: convert to JSON string (as if saved to disk/database) and back
json_str = json.dumps(serialized, default=str)
restored_data = json.loads(json_str)

# Deserialize into a new ProjectState instance
restored_state = ProjectState.deserialize(restored_data)

print(f"Restored session: {restored_state.session_id[:8]}...")
print(f"Project name:     {restored_state.project_name}")
print(f"Tags:             {restored_state.tags}")
print(f"Message history:  {len(restored_state.message_history)} messages (preserved!)")

# Continue the conversation — the agent has full context from the previous session
restored_state.add_message(
    "user",
    "Based on the perovskite challenges you described, which one is closest to "
    "being solved? Give a one-sentence answer."
)

# Create a fresh agent (simulating a new process)
agent2 = SummaryAgent(model=config.llm_model, api_key=config.llm_api_key)
output3 = await agent2.run(restored_state)

print("\nTurn 3 (from restored session):")
print(output3.summary)
print(f"\nMessage history length: {len(restored_state.message_history)}")

Restored session: 3b5cf06b...
Project name:     solar-research
Tags:             ['energy', 'renewables']
Message history:  6 messages (preserved!)

Turn 3 (from restored session):
Among the main perovskite commercialization barriers previously discussed, scalable manufacturing and device performance consistency appear closest to practical resolution, though long-term stability still remains the larger unsolved issue.

Message history length: 8


## 9. Corrupt data handling

What happens when deserialization receives bad data? `BaseSessionState.deserialize()` handles corrupt `message_history` gracefully — it falls back to an empty list and logs a warning. Your custom fields still load correctly.

In [10]:
# Corrupt the message_history with invalid data
corrupt_data = {
    "session_id": "corrupt-session-123",
    "project_name": "still-works",
    "tags": ["resilient"],
    "messages": [{"role": "user", "content": "hello"}],
    "message_history": [{"garbage": True, "not_a_valid_message": 42}],
}

corrupt_state = ProjectState.deserialize(corrupt_data)

print(f"Session ID:      {corrupt_state.session_id}")
print(f"Project name:    {corrupt_state.project_name}")
print(f"Tags:            {corrupt_state.tags}")
print(f"Messages:        {corrupt_state.messages}")
print(f"Message history: {corrupt_state.message_history}  <-- gracefully empty")
print("\nThe state loaded fine — only message_history was lost (with a logged warning).")

2026-04-11 19:22:30.262 | WARNING  | orqest.agents.session_state:deserialize:57 - Failed to deserialize message_history, using empty list


Session ID:      corrupt-session-123
Project name:    still-works
Tags:            ['resilient']
Messages:        [{'role': 'user', 'content': 'hello'}]
Message history: []  <-- gracefully empty

The state loaded fine — only message_history was lost (with a logged warning).


### What's happening under the hood

1. `serialize()` calls `self.model_dump(exclude={"message_history"})` for all Pydantic-native fields, then separately dumps `message_history` via `ModelMessagesTypeAdapter.dump_python(mode="json")`
2. `deserialize()` pops `message_history` from the dict, validates it through `ModelMessagesTypeAdapter.validate_python()`, and passes everything else to the Pydantic constructor
3. If `message_history` validation fails (corrupt data, schema mismatch after upgrade), it catches the exception and falls back to `[]` — the session is usable, just without history
4. Custom fields like `project_name` and `tags` are handled natively by Pydantic and survive the round-trip

---

# Part 3: Compound Tool

The `CompoundTool` pattern separates three concerns:
1. **Agent** — decides what to do (produces structured output via LLM)
2. **Executor** — acts on the decision (runs a function, calls an API, etc.)
3. **State updater** — records the result on state

Hooks fire around the executor step, giving you full observability over the action.

## 10. Define CompoundTool components

In [11]:
from orqest.agents import CompoundTool


# --- Output type: what the review agent produces ---

class ReviewIssue(BaseModel):
    line: int = Field(description="Approximate line number")
    severity: str = Field(description="'critical', 'warning', or 'suggestion'")
    message: str = Field(description="Description of the issue")


class ReviewOutput(BaseModel):
    issues: list[ReviewIssue] = Field(description="Issues found in the code")
    suggestions: list[str] = Field(description="General improvement suggestions")
    overall_quality: str = Field(description="'good', 'needs-work', or 'poor'")


# --- Agent: analyzes code and produces ReviewOutput ---

class CodeReviewAgent(BaseAgent[ProjectState, ReviewOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="code_review",
            system_prompt=(
                "You are a code reviewer. Analyze the provided code snippet and "
                "produce a structured review with specific issues (including line "
                "numbers and severity) and general suggestions. Be constructive."
            ),
            output_type=ReviewOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: ProjectState, **kwargs) -> ReviewOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


# --- Executor: "applies" the review (formats a report) ---

async def execute_review(review: ReviewOutput, state: ProjectState) -> dict[str, Any]:
    """Simulate applying a code review — counts issues and formats a report."""
    critical = sum(1 for i in review.issues if i.severity == "critical")
    warnings = sum(1 for i in review.issues if i.severity == "warning")
    suggestions = sum(1 for i in review.issues if i.severity == "suggestion")
    return {
        "total_issues": len(review.issues),
        "critical": critical,
        "warnings": warnings,
        "suggestions_count": suggestions,
        "overall": review.overall_quality,
        "approved": critical == 0,
    }


# --- State updater: records the review result ---

def update_state_with_review(state: ProjectState, result: dict[str, Any]) -> ProjectState:
    """Record the review result on state."""
    state.add_message("system", f"Code review complete: {result}")
    if "reviewed" not in state.tags:
        state.tags.append("reviewed")
    return state


print("CodeReviewAgent, executor, and state updater defined.")

CodeReviewAgent, executor, and state updater defined.


## 11. Run the compound tool with hooks

Wire everything together: the agent, executor, state updater, and hooks. Then run it on a code snippet.

In [12]:
# Fresh hooks for clean metrics
ct_logging = LoggingHook()
ct_metrics = MetricsHook()
ct_runner = HookRunner(hooks=[ct_logging, ct_metrics])

# Build the compound tool
review_tool = CompoundTool(
    agent=CodeReviewAgent(model=config.llm_model, api_key=config.llm_api_key),
    executor=execute_review,
    state_updater=update_state_with_review,
    hooks=ct_runner,
    name="code_review",
)

# Set up state with a code snippet to review
review_state = ProjectState(project_name="my-api", tags=["python"])
code_snippet = '''\
def get_user(db, user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    data = result.fetchone()
    if data == None:
        return None
    user = {"id": data[0], "name": data[1], "email": data[2], "password": data[3]}
    return user
'''
review_state.add_message("user", f"Review this Python code:\n\n```python\n{code_snippet}```")

print("Running compound tool: agent -> executor -> state update")
print("=" * 60)

agent_output, exec_result = await review_tool.run(review_state, prompt="review code")

print("\n" + "=" * 60)
print("\nAgent output (ReviewOutput):")
print(f"  Overall quality: {agent_output.overall_quality}")
print(f"  Issues found: {len(agent_output.issues)}")
for issue in agent_output.issues:
    print(f"    Line {issue.line} [{issue.severity}]: {issue.message}")
print(f"  Suggestions:")
for s in agent_output.suggestions:
    print(f"    - {s}")

print(f"\nExecutor result:")
print(f"  {exec_result}")

print(f"\nState updated:")
print(f"  Tags: {review_state.tags}")
print(f"  Last system message: {review_state.get_latest_message('system')[:100]}...")

Running compound tool: agent -> executor -> state update
  [LOG] >>> code_review starting | args keys: ['agent_output', 'prompt']
  [LOG] <<< code_review completed in 0.0ms


Agent output (ReviewOutput):
  Overall quality: needs-work
  Issues found: 6
    Line 2 [critical]: SQL query is built with an f-string using user_id directly, which can enable SQL injection if user_id is not strictly validated. Use parameterized queries instead.
    Line 6 [warning]: The returned user object includes the password field. Exposing password data from a general user-fetching function is a security risk; avoid returning it unless absolutely necessary and preferably never return raw passwords.
    Line 5 [suggestion]: Use 'is None' instead of '== None' for idiomatic and more reliable None checks in Python.
    Line 4 [suggestion]: No error handling is present around database execution/fetching. Database failures will propagate unexpectedly; consider handling exceptions or documenting them clearly.
    

## 12. Assessment — metrics collected by hooks

The `MetricsHook` silently collected timing and count data throughout the compound tool run. Let's see what it captured.

In [13]:
print("Metrics from the compound tool run:\n")
ct_metrics.report()

print("\nThe hooks observed the executor step without any coupling to the")
print("executor or agent code. You could swap in a DataDog hook, a")
print("Prometheus hook, or any other observer — the CompoundTool doesn't change.")

Metrics from the compound tool run:

--- Metrics Report ---
  code_review: 1 calls, 0.0ms total, 0.0ms avg, 0 errors

The hooks observed the executor step without any coupling to the
executor or agent code. You could swap in a DataDog hook, a
Prometheus hook, or any other observer — the CompoundTool doesn't change.


### What's happening under the hood

1. `CompoundTool.run(state, prompt)` first calls `agent.run(state)` to get the structured `ReviewOutput` from the LLM
2. It fires `hooks.fire_before()` with the agent output and prompt as args
3. It calls `executor(agent_output, state)` — your async function that acts on the decision
4. On success, it fires `hooks.fire_after()` with the result and measured duration
5. On failure, it fires `hooks.fire_error()` with the exception — then re-raises it
6. Finally, if a `state_updater` is provided, it calls `state_updater(state, result)` to record the outcome
7. Returns `(agent_output, execution_result)` so the caller has access to both

The key insight: **the agent never knows about the executor, and the executor never knows about the hooks**. Each concern is cleanly separated.